In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

In [3]:
BASE = Path("/content/drive/MyDrive/WasteScope")
REPORTS = BASE / "reports" / "tracking"
OUT = REPORTS / "final"
OUT.mkdir(parents=True, exist_ok=True)

RUNS = [
    "T0_bytetrack", "T1_bytetrack_calibrated",
    "T2_bytetrack_postprocessed", "T3_botsort",
]
SELECTED_RUN = "T3_botsort"
sources = [REPORTS / run / "val" for run in RUNS]
sources.append(REPORTS / SELECTED_RUN / "test")

comparison = pd.concat(
    [pd.read_csv(folder / "overall.csv") for folder in sources],
    ignore_index=True
)

In [4]:
comparison["RawTracks"] = comparison["RawTracks"].fillna(comparison["Predicted"]).astype(int)
comparison["Report"] = [f"../{folder.parent.name}/{folder.name}/README.md" for folder in sources]

selected_folder = REPORTS / SELECTED_RUN / "val"
configuration = json.loads((selected_folder / "evaluation_config.json").read_text())
test_sequences = pd.read_csv(
    REPORTS / SELECTED_RUN / "test" / "per_sequence.csv",
    dtype={"Sequence": str},
).set_index("Sequence")
view = ["Run", "Split", "HOTA", "IDF1", "MOTA", "IDSW", "MAE", "SAD"]

display(comparison[view].round(4))

,Run,Split,HOTA,IDF1,MOTA,IDSW,MAE,SAD
0,T0_bytetrack,val,60.2081,75.0568,62.3903,38,3.6923,48
1,T1_bytetrack_calibrated,val,62.1487,78.6632,63.1801,30,2.2308,29
2,T2_bytetrack_postprocessed,val,62.2186,78.7734,63.4082,17,1.0769,14
3,T3_botsort,val,67.3233,80.9069,67.1639,27,1.0769,14
4,T3_botsort,test,66.2731,78.3097,64.9351,32,0.5714,8


In [5]:
detector = configuration["detector"]
selected_config = {
    "status": "evaluation_complete",
    "selected_run": SELECTED_RUN,
    "selected_on": "val",
    "selection_reason": "Highest validation HOTA and IDF1; counting MAE tied with T2. T2 has fewer ID switches.",
    "detector": {
        "run": detector["detector"],
        "weights": f"runs/detect/{detector['detector']}/weights/best.pt",
        "checkpoint_sha256": detector["checkpoint_sha256"],
        "inference": {key: detector[key] for key in (
            "imgsz", "confidence", "max_det", "half", "rect", "nms"
        )},
    },
    "tracker": configuration["tracker"],
    "tracker_frame_rate_argument": configuration["tracker_frame_rate_argument"],
    "postprocessing": configuration["postprocessing"],
    "data_config": "prepared/tracking/data_config.json",
    "annotations_file": f"prepared/tracking/{configuration['annotations_file']}",
    "annotations_sha256": configuration["annotations_sha256"],
    "manifest_sha256": configuration["manifest_sha256"],
    "evaluation": configuration["evaluation"],
    "versions": configuration["versions"],
    "reports": {split: f"reports/tracking/{SELECTED_RUN}/{split}" for split in ("val", "test")}
}

In [7]:
def markdown_table(frame):
    lines = ["| " + " | ".join(frame.columns) + " |",
             "| " + " | ".join(["---"] * len(frame.columns)) + " |"]
    lines.extend("| " + " | ".join(row) + " |"
                 for row in frame.round(4).astype(str).itertuples(index=False, name=None))
    return "\n".join(lines)

validation = comparison[comparison["Split"].eq("val")]
test = comparison[comparison["Split"].eq("test")]
exact_counts = int(test_sequences["CountError"].eq(0).sum())
failure_ids = test_sequences["HOTA"].nsmallest(2).index.union(
    test_sequences["IDSW"].nlargest(1).index, sort=False
)
limitations = test_sequences.loc[failure_ids, [
    "GT", "RawTracks", "Predicted", "HOTA", "IDF1", "MOTA", "IDSW"
]].reset_index()
links = "\n".join(f"- [{row.Run} / {row.Split}]({row.Report})"
                  for row in comparison.itertuples(index=False))

readme = f"""# Tracking final report

Selected configuration: **{SELECTED_RUN}**, chosen on validation before the final test.
T3 has the highest validation HOTA and IDF1, with counting MAE tied with T2.
T2 has fewer identity switches; this is the selection trade-off.

## Validation comparison

{markdown_table(validation[view])}

## Final held-out test

{markdown_table(test[view])}

Exact sequence counts: **{exact_counts}/{len(test_sequences)}**.
GT tracks: **{int(test['GT'].iloc[0])}**; counted tracks: **{int(test['Predicted'].iloc[0])}**.
Sum of absolute per-sequence count errors: **{int(test['SAD'].iloc[0])}**.
Settings were frozen before the test; the test was not used for tuning.

## Measured limitations

The two lowest-HOTA test sequences and the sequence with the most ID switches:

{markdown_table(limitations)}

Count agreement can coexist with poor tracking quality. These results document remaining errors without assigning unverified causes.

## Protocol and counting

- Class-aware TrackEval: HOTA IoU 0.05:0.05:0.95; IDF1/MOTA IoU 0.50.
- HOTA, DetA, AssA, IDF1, MOTA and MAPE are percentages. IDSW is a count.
- Counting metrics measure unique objects per sequence. RawTracks counts IDs after reconnection; Predicted counts IDs meeting the minimum-duration rule.
- Minimum duration is 10 distinct observed frames, after same-class reconnection with at most 5 missing frame steps and normalized center distance 0.05.
- Tracking metrics retain all emitted boxes; duration filtering affects counting only. Counts are finalized after the complete sequence.
- MAE/RMSE/SAD/MAPE use per-sequence count errors. The difference between total GT and total predicted counts can hide errors that cancel.
- Results use the corrected annotations and frozen video split. These are single-run results.
- Saved tracking wall times cover tracker/postprocessing, with detections already cached; they are not end-to-end pipeline FPS.

## Saved configuration and sources

- [Selected configuration](selected_config.json): detector inference settings, tracker, postprocessing, dependency versions, and recorded data/checkpoint hashes. File paths are relative to the WasteScope Drive root.
- [Full comparison](comparison.csv)
{links}
"""

comparison.to_csv(OUT / "comparison.csv", index=False)
(OUT / "selected_config.json").write_text(json.dumps(selected_config, indent=2) + "\n", encoding="utf-8")
(OUT / "README.md").write_text(readme, encoding="utf-8")

3205